# Data augmentation

In [6]:
import albumentations as A
from torchvision import transforms
import cv2
import os
import glob
import numpy as np

In [7]:
# Define the Path to the Data
INPUT_IMG_DIR = "../0_datasets/labelled_images/images"
INPUT_LBL_DIR = "../0_datasets/labelled_images/labels"
OUTPUT_DIR = "augmented_dataset"

os.makedirs(f"{OUTPUT_DIR}/images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels", exist_ok=True)

In [8]:
transform = A.Compose([
    # Geometric
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=25, p=0.6, border_mode=cv2.BORDER_REFLECT),
    A.Affine(scale=(0.8, 1.2), p=0.5),
    A.Perspective(scale=(0.05, 0.1), p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2,
                       rotate_limit=15, p=0.5,
                       border_mode=cv2.BORDER_REFLECT),
    # Blur / noise
    A.GaussNoise(p=0.3),
    A.ISONoise(p=0.2),
    A.GaussianBlur(blur_limit=3, p=0.3),
    A.MotionBlur(blur_limit=3, p=0.2),
    A.MedianBlur(blur_limit=3, p=0.1),
    A.Defocus(radius=(1, 3), p=0.2),
    # Lighting / colour
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    A.CLAHE(p=0.3),
    A.RandomSunFlare(p=0.1),
    A.RandomRain(p=0.1),
    A.RandomFog(p=0.1),
    A.RandomShadow(p=0.2),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30,
                         val_shift_limit=0, p=0.3),
    A.ToGray(p=0.1),
    A.ChannelShuffle(p=0.1),
    A.RGBShift(r_shift_limit=20, g_shift_limit=20, b_shift_limit=20, p=0.2),
    # Compression / distortion
    A.ImageCompression(quality_range=(75, 100), p=0.3),   # fixed arg name
    A.ElasticTransform(alpha=1, sigma=50, p=0.2),          # removed alpha_affine
    A.GridDistortion(p=0.2),
    A.OpticalDistortion(distort_limit=0.05, p=0.2),        # removed shift_limit
    # Cutout
    A.CoarseDropout(num_holes_range=(1, 4),                # fixed arg names
                    hole_height_range=(10, 20),
                    hole_width_range=(10, 20), p=0.2),
],
bbox_params=A.BboxParams(
    format='yolo',           # ← keep YOLO format end-to-end, no conversion needed
    label_fields=['class_labels'],
    min_area=0.0,
    min_visibility=0.1,
    clip=True,               # ← clips boxes to image boundary instead of crashing
))

In [9]:
MIN_BOX_NORM = 0.001   # drop boxes whose normalised w or h < 0.1 % of image


def parse_yolo_label(label_path):
    """Return (class_labels, bboxes_yolo). Skips degenerate boxes."""
    class_labels, bboxes = [], []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls  = int(float(parts[0]))
            cx, cy, w, h = map(float, parts[1:5])
            # Skip boxes that are essentially zero-size (the root cause of your errors)
            if w < MIN_BOX_NORM or h < MIN_BOX_NORM:
                continue
            # Clamp to valid range
            cx = np.clip(cx, 0.0, 1.0)
            cy = np.clip(cy, 0.0, 1.0)
            w  = np.clip(w,  0.0, 1.0)
            h  = np.clip(h,  0.0, 1.0)
            class_labels.append(cls)
            bboxes.append([cx, cy, w, h])
    return class_labels, bboxes


def is_valid_box(cx, cy, w, h):
    return w >= MIN_BOX_NORM and h >= MIN_BOX_NORM

In [10]:
image_files = glob.glob(f"{INPUT_IMG_DIR}/*.jpg") + \
              glob.glob(f"{INPUT_IMG_DIR}/*.jpeg")

print(f"Found {len(image_files)} images. Starting augmentation…")
print("=" * 70)

stats = dict(total=0, augmented=0, skipped=0,
             orig_boxes=0, aug_boxes=0, lost_boxes=0)

for img_path in image_files:
    base_name  = os.path.basename(img_path)
    file_name  = os.path.splitext(base_name)[0]
    label_path = os.path.join(INPUT_LBL_DIR, f"{file_name}.txt")

    image = cv2.imread(img_path)
    if image is None:
        print(f"❌ Cannot read image: {file_name}")
        stats['skipped'] += 1
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    if not os.path.exists(label_path):
        print(f"⚠️  No label file for {file_name}")
        stats['skipped'] += 1
        continue

    class_labels, bboxes = parse_yolo_label(label_path)

    if not bboxes:
        print(f"⚠️  No valid bboxes in {file_name}.txt (all were degenerate)")
        stats['skipped'] += 1
        continue

    stats['total']      += 1
    stats['orig_boxes'] += len(bboxes)
    successful = 0

    for i in range(100):
        try:
            result      = transform(image=image, bboxes=bboxes,
                                    class_labels=class_labels)
            aug_img     = result['image']
            aug_bboxes  = result['bboxes']
            aug_labels  = result['class_labels']

            # Extra guard: discard any box still too small after transform
            valid = [(b, l) for b, l in zip(aug_bboxes, aug_labels)
                     if is_valid_box(*b)]

            if not valid:
                stats['lost_boxes'] += len(aug_bboxes)
                continue

            stats['lost_boxes'] += len(aug_bboxes) - len(valid)
            valid_bboxes, valid_labels = zip(*valid)

            aug_name = f"{file_name}_aug_{i}"

            cv2.imwrite(f"{OUTPUT_DIR}/images/{aug_name}.jpg",
                        cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))

            with open(f"{OUTPUT_DIR}/labels/{aug_name}.txt", "w") as f:
                for lbl, (cx, cy, w, h) in zip(valid_labels, valid_bboxes):
                    # Values are already normalised YOLO coords; just clamp
                    cx = np.clip(cx, 0.0, 1.0)
                    cy = np.clip(cy, 0.0, 1.0)
                    w  = np.clip(w,  0.0, 1.0)
                    h  = np.clip(h,  0.0, 1.0)
                    f.write(f"{lbl} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

            stats['aug_boxes'] += len(valid_bboxes)
            successful += 1

        except Exception as e:
            print(f"   ⚠️  Iter {i} error: {e}")
            continue   # skip bad iteration, don't break the whole image

    stats['augmented'] += 1
    print(f"✅ {file_name}: {successful}/100 augmentations")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("📊 AUGMENTATION SUMMARY")
print("=" * 70)
print(f"  Images processed : {stats['total']}")
print(f"  Images skipped   : {stats['skipped']}")
print(f"  Original boxes   : {stats['orig_boxes']}")
print(f"  Augmented boxes  : {stats['aug_boxes']}")
print(f"  Lost boxes       : {stats['lost_boxes']}")
ratio = stats['aug_boxes'] / max(stats['orig_boxes'] * 100, 1) * 100
print(f"  Box preservation : {ratio:.1f} %")
print("=" * 70)
print(f"✅ Done — outputs in '{OUTPUT_DIR}'")

Found 51 images. Starting augmentation…
✅ Night_MA111_jpg.rf.AcgugP02XsoSoNCkvwRD: 99/100 augmentations
✅ MA153_jpeg.rf.swIZYao54PErIlAmzBug: 100/100 augmentations
✅ Night_MA113_jpeg.rf.zJ9fvzWMHPdT9vikOete: 99/100 augmentations
✅ MA114_jpeg.rf.zj3P9wbU7oJKCOShsDku: 88/100 augmentations
✅ MA106_jpeg.rf.T1Q1gI1TkIM3FjPYPU8S: 91/100 augmentations
✅ MA158_jpeg.rf.qozl118OFzC3M5Ja33cM: 100/100 augmentations
✅ MA115_jpeg.rf.yfDDIv7Wo2wS0tcfsxG3: 88/100 augmentations
✅ MA111_jpeg.rf.u9rPSFaY6N5fWYDxTdAv: 89/100 augmentations
✅ Night_MA105_jpeg.rf.DEeKd6ZDu7qbvvFzgW1y: 100/100 augmentations
✅ Night_MA104_jpeg.rf.Wgmhiyw0JLenLEuLqT9r: 96/100 augmentations
✅ Night_MA110_jpeg.rf.N1H7Ueu3Ge2yE9oMrBWr: 89/100 augmentations
✅ MA150_jpeg.rf.ucsM76IgV8VbJlQv9FF0: 99/100 augmentations
✅ Night_MA107_jpeg.rf.vDOMouRhWmic0nv8q8ou: 83/100 augmentations
✅ MA113_jpeg.rf.99JN0FDclebqWAJrqTd0: 100/100 augmentations
✅ Night_MA112_jpeg.rf.vRK4OA0BUYCMH6l7OtfT: 96/100 augmentations
✅ Night_MA102_jpeg.rf.pp5sxw5R

# Split the data into train, validation and testing sets

In [11]:
import os
import random
import shutil

# Set your paths
dataset_path = "augmented_dataset"
output_path = "split_dataset"

# Split Ratios
train_ratio = 0.7
val_ratio = 0.2
# Test ratio will be the remaining 0.1

def split_dataset():
    # Create the YOLO directory structure
    for folder in ['images', 'labels']:
        for split in ['train', 'val', 'test']:
            os.makedirs(os.path.join(output_path, folder, split), exist_ok=True)

    # Get all image filenames (assumes .jpg, change if using .png)
    all_imgs = [f for f in os.listdir(os.path.join(dataset_path, "images")) if f.endswith('.jpg')]
    random.shuffle(all_imgs)

    # Calculate split indices
    train_count = int(len(all_imgs) * train_ratio)
    val_count = int(len(all_imgs) * val_ratio)

    train_files = all_imgs[:train_count]
    val_files = all_imgs[train_count:train_count + val_count]
    test_files = all_imgs[train_count + val_count:]

    def move_files(file_list, split_name):
        for filename in file_list:
            basename = os.path.splitext(filename)[0]
            
            # Move Image
            shutil.copy(os.path.join(dataset_path, "images", filename), 
                        os.path.join(output_path, "images", split_name, filename))
            
            # Move Label (matching the image name)
            label_name = f"{basename}.txt"
            if os.path.exists(os.path.join(dataset_path, "labels", label_name)):
                shutil.copy(os.path.join(dataset_path, "labels", label_name), 
                            os.path.join(output_path, "labels", split_name, label_name))

    move_files(train_files, 'train')
    move_files(val_files, 'val')
    move_files(test_files, 'test')
    
    print(f"✅ Split Complete: {len(train_files)} Train | {len(val_files)} Val | {len(test_files)} Test")

split_dataset()

✅ Split Complete: 3440 Train | 983 Val | 492 Test
